# STEP 4 — RQ2: SHAP explainability + cross-set driver consistency

Explained model: **RAW / natural-distribution LightGBM** (NO balancing, NO calibration) —
the model the profit chain will use. Fit on ALL data with Step 2 `best_params_<set>`
(seed=42). Explainer: `shap.TreeExplainer` (exact TreeSHAP).

5 **labeled** sets **independent**; never merged. The main contribution: comparing, at the
**concept level**, whether the same **concepts** (tenure, contract, usage, decline,
complaint/support, monetary, payment, service, demographics) are strong churn drivers
across different sectors too (without merging data, via the `src/concept_map.py` map).
Heavy logic in `src/`.

In [1]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd


def _bul_kok():
    for c in [Path.cwd(), *Path.cwd().parents]:
        if (c / "config.yaml").exists():
            return c
    raise RuntimeError("config.yaml not found")


KOK = _bul_kok()
if str(KOK) not in sys.path:
    sys.path.insert(0, str(KOK))

warnings.filterwarnings("ignore")
from src import concept_map as km
from src import config as cfg
from src import plotstyle as ps
from src import shap_analysis as sa
from src import strings as S

np.random.seed(cfg.SEED)
ps.uygula()
cfg.klasorleri_hazirla()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

CIKTI = []


def yaz(s=""):
    print(s)
    CIKTI.append(str(s))

## 1. Data loading

In [2]:
veriler = {k: pd.read_csv(cfg.PROCESSED / f"{k}_clean.csv") for k in cfg.DATASETS}
for k, d in veriler.items():
    yaz(f"{k:11s} {d.shape}")

telco       (7043, 20)
cell2cell   (51047, 57)
ecommerce   (3941, 11)
iranian     (3150, 14)
bank        (10000, 11)


## 2. SHAP for each set: global importance, beeswarm, dependence, individual (waterfall)
Since cell2cell is large, a stratified sample (n<=15000) is used for the explanation.

In [3]:
yaz(S.MSG["bolum"].format(ad="SHAP COMPUTATION"))
tum_onem = {}
tum_local = {}
for k in cfg.DATASETS:
    t = time.time()
    h = sa.hazirla(k, veriler[k], cfg.SEED)
    sv, evb = sa.shap_hesap(h)
    onem = sa.onem_orijinal(sv, h["orijinaller"])
    tum_onem[k] = onem
    sa.tablo_global(k, onem)
    # unmapped (not in the map) features
    eslenemeyen = [f for f in onem.index if f not in km.HARITA]
    if eslenemeyen:
        yaz(S.MSG4["eslenemeyen"].format(set=k, liste=eslenemeyen))
    # global figures
    sa.figur_beeswarm(k, sv, h)
    sa.figur_importance(k, onem)
    sa.figur_dependence(k, sv, h, onem)
    # individual: highest / lowest probability customer
    hi = int(np.argmax(h["proba"]))
    lo = int(np.argmin(h["proba"]))
    sa.figur_waterfall(k, sv, evb, h, hi, "high")
    sa.figur_waterfall(k, sv, evb, h, lo, "low")
    lokal = sa.tablo_local(k, sv, h, hi, lo)
    tum_local[k] = {"hi_p": float(h["proba"][hi]), "lo_p": float(h["proba"][lo])}
    if h["N"] > h["n_ornek"]:
        yaz(S.MSG4["ornek_not"].format(set=k, n=h["n_ornek"], N=h["N"]))
    yaz(S.MSG4["set"].format(set=k, n=h["n_ornek"], top=list(onem.head(5).index)))
    yaz(f"  ({time.time()-t:.0f}s) probability: high={h['proba'][hi]:.3f} low={h['proba'][lo]:.3f}")

===== SHAP COMPUTATION =====


telco: SHAP computed (sample n=7043); top drivers: ['Contract', 'tenure', 'MonthlyCharges', 'OnlineSecurity', 'InternetService']
  (4s) probability: high=0.908 low=0.011


cell2cell: stratified sample n=15000 for SHAP explanation (full data 51047).
cell2cell: SHAP computed (sample n=15000); top drivers: ['CurrentEquipmentDays', 'MonthlyMinutes', 'MonthsInService', 'PercChangeMinutes', 'CreditRating']
  (12s) probability: high=0.816 low=0.048


ecommerce: SHAP computed (sample n=3941); top drivers: ['Tenure', 'Complain', 'CashbackAmount', 'NumberOfAddress', 'MaritalStatus']
  (7s) probability: high=1.000 low=0.000


iranian: SHAP computed (sample n=3150); top drivers: ['Status', 'Frequency of use', 'Complains', 'Seconds of Use', 'Call  Failure']
  (7s) probability: high=1.000 low=0.000


bank: SHAP computed (sample n=10000); top drivers: ['Age', 'NumOfProducts', 'IsActiveMember', 'Balance', 'Geography']
  (3s) probability: high=0.947 low=0.019


## 3. iranian 'Status' SHAP follow-up
Does Status behave like a normal predictor, or is it a dominance that overrides everything on its own?

In [4]:
onem_ir = tum_onem["iranian"]
sira = list(onem_ir.index).index("Status") + 1
pay = 100 * onem_ir["Status"] / onem_ir.sum()
yorum = "not solely dominant, together with other usage drivers" if pay < 40 else "suspected dominance"
yaz(S.MSG["bolum"].format(ad="iranian 'Status' SHAP"))
yaz(S.MSG4["iranian_status"].format(sira=sira, toplam=len(onem_ir), pay=pay, yorum=yorum))

===== iranian 'Status' SHAP =====
iranian SHAP: 'Status' global importance rank=1/13, share=18.3% — not solely dominant, together with other usage drivers


## 4. Cross-set conceptual driver consistency (MAIN CONTRIBUTION)
Each set's driver importances are aggregated into concepts (share), concept × set heatmap.

In [5]:
matris, paylar = sa.tutarlilik(tum_onem)
yol_heat = sa.figur_tutarlilik(matris, list(cfg.DATASETS.keys()))
yaz(S.MSG["bolum"].format(ad="CONCEPTUAL CONSISTENCY (concept share, per set)"))
yaz(matris.to_string(index=False))
yaz(S.MSG["kayit"].format(yol=cfg.TABLES / "rq2_driver_consistency.csv"))
yaz(S.MSG["kayit"].format(yol=yol_heat))

===== CONCEPTUAL CONSISTENCY (concept share, per set) =====
                              Concept  telco  cell2cell  ecommerce  iranian   bank  Top-3 sector count
       Relationship duration (tenure) 0.1062     0.0904     0.2368   0.0810 0.0104                   1
                Contract / commitment 0.3580     0.0000     0.0000   0.0000 0.0000                   1
                         Usage volume 0.0000     0.2148     0.0471   0.5570 0.1372                   3
                        Usage decline 0.0000     0.0978     0.0000   0.0000 0.0000                   0
                 Engagement / recency 0.0000     0.0000     0.2400   0.0000 0.0000                   1
Complaint / support / service quality 0.0566     0.0931     0.1920   0.2287 0.0000                   2
            Monetary value / spending 0.1178     0.0626     0.1187   0.0980 0.1078                   2
                        Credit / risk 0.0000     0.0504     0.0000   0.0000 0.0155                   0
             

## 5. Summary
Top-5 drivers per set; consistent vs sector-specific concepts. Interpretation/decision left to the user.

In [6]:
yaz(S.MSG["bolum"].format(ad="SUMMARY"))
for k in cfg.DATASETS:
    yaz(f"{k:11s} top-5: {list(tum_onem[k].head(5).index)}")
tutarli = matris[matris[S.KOLON4["top3_say"]] >= 3][S.KOLON4["kavram"]].tolist()
ozgu = matris[matris[S.KOLON4["top3_say"]] == 1][S.KOLON4["kavram"]].tolist()
yaz(f"\nCONSISTENT (top-3 in >=3 sectors): {tutarli}")
yaz(f"SECTOR-SPECIFIC (top-3 in only 1 sector): {ozgu}")
yaz("")
yaz(S.MSG4["bitti"])

_log = cfg.LOGS / "adim4_ozet.log"
_log.write_text("\n".join(CIKTI) + "\n", encoding="utf-8")
print(S.MSG["kayit"].format(yol=_log))

===== SUMMARY =====
telco       top-5: ['Contract', 'tenure', 'MonthlyCharges', 'OnlineSecurity', 'InternetService']
cell2cell   top-5: ['CurrentEquipmentDays', 'MonthlyMinutes', 'MonthsInService', 'PercChangeMinutes', 'CreditRating']
ecommerce   top-5: ['Tenure', 'Complain', 'CashbackAmount', 'NumberOfAddress', 'MaritalStatus']
iranian     top-5: ['Status', 'Frequency of use', 'Complains', 'Seconds of Use', 'Call  Failure']
bank        top-5: ['Age', 'NumOfProducts', 'IsActiveMember', 'Balance', 'Geography']

CONSISTENT (top-3 in >=3 sectors): ['Usage volume']
SECTOR-SPECIFIC (top-3 in only 1 sector): ['Relationship duration (tenure)', 'Contract / commitment', 'Engagement / recency', 'Device / equipment']

STEP 4 (RQ2) complete. Interpretation left to the user. No profit/ROI (RQ3) performed.
Saved: /Users/emrahfidan/Desktop/churn-xai-profit/outputs/logs/adim4_ozet.log
